# STRATA, kernel do tipo D na velocidade do P: Llama-3.1-8B

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*. Leva ~1h30 a 2h (duas compilações do llama.cpp, três gravações do STRATA, KL e velocidade).

Os tipos D (`STRATA_Q4_K_D1/D2`, `STRATA_Q6_K_D1/D2`) passaram na qualidade com os bits do GPTQ (PR #15), mas geravam a 0,65 da velocidade dos tipos P (seção 30 de `docs/05-resultados.md`). O kernel novo (`docs/11-kernel-do-tipo-d.md`) é o kernel do P mais uma consulta, liga a fusão gate+up e tem um caminho sem classe para tabelas "linha" (as três classes iguais).

O que este notebook mede, na mesma sessão:
1. **Conferência do CUDA**: `strata_cuda_check` (GPU contra CPU contra float64), agora com uma tabela linha em cada tipo D.
2. **Velocidade** de geração (tg128) do llama.cpp **antes** (kernel antigo) e **depois** (este PR), com os mesmos GGUFs: P, D e D linha nos níveis 1 e 2, Q5_K_M e Q6_K com imatrix. Três rodadas intercaladas, mediana. O "depois" roda também sem fusão, para separar os ganhos.
3. **Qualidade**: a KL do D pelo MMVQ (`-ub 8`) antes e depois (tem que bater) e a KL do D linha contra a do D (caminho da desquantização).

**Régua (escrita antes de medir, `docs/11`):**
- velocidade, para o D exato e para o D linha: **passa** se D ≥ 0,95 × P e D ≥ 0,95 × k-quant (Q5_K_M no nível 1, Q6_K no nível 2) nos dois níveis; **zona cinza** se a menor razão ficar entre 0,85 e 0,95; **reprova** se D < 0,85 × P em algum nível.
- qualidade: D antes = D depois dentro do ±; D linha passa se KL ≤ 1,03 × a do D nos dois níveis.

Ocupa ~90 GB de disco no pico.

In [ ]:
KLD_CHUNKS = 80            # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125       # trechos de 512 tokens do wikitext de treino para a imatrix
RODADAS = 3                # rodadas intercaladas de velocidade (mediana)
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"

In [ ]:
import os, re, json, shutil, subprocess, statistics
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100.'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if 'A100' not in GPU:
    print('Aviso: a régua foi escrita para o A100; em outra GPU os números não se comparam com a seção 30.')
if DISCO < 100:
    print('Aviso: pouco disco. Ponha KLD_CHUNKS menor.')

Dois llama.cpp no mesmo commit: **antes** com o patch do STRATA no commit `f441834` (kernel D antigo, só o patch consertado) e **depois** com o patch deste PR.

In [ ]:
STRATA_SHA = "9cfb789ceffebd34a86ae952cf194b7a521fb5c4"   # este PR: kernel D novo, tabela linha, fusão
ANTES_SHA = "f441834ff0e0e69d29c352725113bd419e982734"    # o kernel D antigo (só o patch do main consertado)
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"    # o commit em que o patch dos tipos foi feito
%cd /content
def clona(pasta, url, sha):
    !rm -rf {pasta} && git init -q {pasta} && cd {pasta} && git remote add origin {url} && git fetch -q --depth 1 origin {sha} && git checkout -q FETCH_HEAD
clona('strata', 'https://github.com/Cantyll/strata.git', STRATA_SHA)
clona('strata_antes', 'https://github.com/Cantyll/strata.git', ANTES_SHA)
CM = '-DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON'
for pasta, st, alvos in [('llama.cpp', 'strata', 'llama-quantize llama-perplexity llama-imatrix llama-bench'),
                         ('llama_antes', 'strata_antes', 'llama-perplexity llama-bench')]:
    clona(pasta, 'https://github.com/ggml-org/llama.cpp.git', LLAMA_SHA)
    !sh {st}/kernel/ggml/aplicar.sh {pasta}
    # o fim do log da compilação fica visível, para aparecer qualquer erro
    !cd {pasta} && cmake -B build {CM} 2>&1 | tail -2 && cmake --build build -j $(nproc) --target {alvos} 2>&1 | tail -5
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
LB, LA = '/content/llama.cpp/build/bin', '/content/llama_antes/build/bin'
for b in (LB, LA):
    assert os.path.exists(f'{b}/llama-bench') and os.path.exists(f'{b}/llama-perplexity'), f'compilação falhou em {b}'
print('dois llama.cpp prontos')

Conferência do CUDA: cada tipo P e D (tabela qualquer e tabela linha) multiplicado por ativações com 1 a 512 colunas, na GPU e no CPU. Tem que terminar em `tudo OK`.

In [ ]:
!cc -O2 strata/tests/kernel/strata_cuda_check.c -I llama.cpp/ggml/include -L {LB} -lggml -lggml-base -lggml-cpu -lggml-cuda -lm -Wl,-rpath,{LB} -o strata_cuda_check
out = !./strata_cuda_check
print('\n'.join(l for l in out if 'DIVERGE' in l or 'linha' in l or 'N=1 ' in l or l is out[-1]))
CUDA_OK = out[-1].strip() == 'tudo OK'
assert CUDA_OK, 'O CUDA diverge do CPU: cole a saída acima na thread.'

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste')

Modelo em BF16, imatrix, k-quants com imatrix e a base da KL. O BF16 é apagado depois da base.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!{LB}/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
for q in ['Q4_K_M', 'Q5_K_M', 'Q6_K']:
    !{LB}/llama-quantize --imatrix {G}/imatrix.gguf {G}/BF16.gguf {G}/{q}_imat.gguf {q} > {G}/{q}_imat.log 2>&1
!{LB}/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
!rm -f {G}/BF16.gguf
!ls -la {G}/*.gguf /content/kld_base.bin
!df -h /content | tail -1

O STRATA sobre o mesmo Q4_K_M com imatrix: tipos P, D (tabela com classes, a do PR #15) e D linha (`--sem-classe`), níveis 1 e 2. Os bits aqui são os do Lloyd (o mais perto), não os do GPTQ: a velocidade não depende deles, e a comparação de qualidade D × D linha é pareada.

In [ ]:
%cd /content/strata
NIV = f'python tests/kernel/strata_gguf_niveis.py --model {src} --gguf {G}/Q4_K_M_imat.gguf --nivel 1 2'
!{NIV} --out {G}/P_N1.gguf {G}/P_N2.gguf 2>&1 | grep -v Warning | tail -2
!{NIV} --sub-celulas --imatrix {G}/imatrix.gguf --out {G}/D_N1.gguf {G}/D_N2.gguf 2>&1 | grep -v Warning | tail -2
!{NIV} --sub-celulas --sem-classe --imatrix {G}/imatrix.gguf --out {G}/DL_N1.gguf {G}/DL_N2.gguf 2>&1 | grep -v Warning | tail -2
%cd /content
!rm -rf {src}
!ls -la {G}/*.gguf
!df -h /content | tail -1

Funções de medida. `kl_de` com `ub=8` força o produto escalar MMVQ (o caminho da geração); sem ele, o prompt passa pela desquantização + cuBLAS.

In [ ]:
kld, tam, vel = {}, {}, {}
def kl_de(path, nome, bindir=LB, ub=None):
    extra = f'-b 512 -ub {ub}' if ub else ''
    out = !{bindir}/llama-perplexity -m {path} --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 {extra} 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {nome}:', '\n'.join(out[-8:]))
        return
    kld[nome] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    tam[nome] = os.path.getsize(path) / 2**30
    print(f'{nome}: KL ×1000 {1000 * kld[nome][0]:.3f} ± {1000 * kld[nome][1]:.3f}, mesmo 1º token {kld[nome][2]}%, {tam[nome]:.3f} GiB')

def tg(path, bindir=LB, env=''):
    # tok/s de geração (tg128), 5 repetições numa chamada
    out = !{env} {bindir}/llama-bench -m {path} -ngl 99 -p 0 -n 128 -r 5 -o json 2>/dev/null
    txt = '\n'.join(out)
    try:
        return json.loads(txt[txt.index('['):txt.rindex(']') + 1])[0]['avg_ts']
    except Exception as e:
        print('Erro no llama-bench:', path, e, '\n'.join(out[-5:]))
        return float('nan')

Qualidade. KL pela desquantização (P, D, D linha e os k-quants pelo MMQ, de referência) e pelo MMVQ (`-ub 8`): D antes e depois, D linha e P depois.

In [ ]:
for q in ['Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat']:
    kl_de(f'{G}/{q}.gguf', q)
for n in ['P_N1', 'P_N2', 'D_N1', 'D_N2', 'DL_N1', 'DL_N2']:
    kl_de(f'{G}/{n}.gguf', n)
for n in ['D_N1', 'D_N2']:
    kl_de(f'{G}/{n}.gguf', f'{n}_mmvq_antes', bindir=LA, ub=8)
for n in ['P_N1', 'P_N2', 'D_N1', 'D_N2', 'DL_N1', 'DL_N2']:
    kl_de(f'{G}/{n}.gguf', f'{n}_mmvq', ub=8)

Velocidade: todas as combinações numa rodada, depois a próxima (o A100 do Colab varia com o tempo; intercalar evita que uma deriva caia só num lado). Mediana das rodadas.

In [ ]:
SEM = 'GGML_CUDA_DISABLE_FUSION=1'
casos = [(m, LB, '', m) for m in ['Q5_K_M_imat', 'Q6_K_imat', 'P_N1', 'P_N2', 'D_N1', 'D_N2', 'DL_N1', 'DL_N2']]
casos += [(m + '_sem_fusao', LB, SEM, m) for m in ['P_N1', 'P_N2', 'D_N1', 'D_N2', 'DL_N1', 'DL_N2']]
casos += [(m + '_antes', LA, '', m) for m in ['P_N1', 'P_N2', 'D_N1', 'D_N2']]
rodadas = {c[0]: [] for c in casos}
for r in range(RODADAS):
    for nome, b, env, m in casos:
        rodadas[nome].append(tg(f'{G}/{m}.gguf', b, env))
    print(f'rodada {r + 1}:', {k: round(v[-1], 1) for k, v in rodadas.items()})
vel = {k: statistics.median(v) for k, v in rodadas.items()}

**O veredito**, pela régua de `docs/11`.

In [ ]:
import math
from IPython.display import Markdown, display
KQ = {1: 'Q5_K_M_imat', 2: 'Q6_K_imat'}

def razoes(pref, suf=''):
    # [N1/P, N1/kq, N2/P, N2/kq] do candidato pref (D ou DL)
    r = []
    for k in (1, 2):
        d = vel.get(f'{pref}_N{k}{suf}', float('nan'))
        r += [d / vel.get(f'P_N{k}{suf}', float('nan')), d / vel[KQ[k]]]
    return r

def veredito(r):
    if any(math.isnan(x) for x in r):
        return 'sem dado'
    if min(r[0], r[2]) < 0.85:
        return 'reprova'
    return 'passa' if min(r) >= 0.95 else 'zona cinza'

def kl(n):
    return kld[n][0] if n in kld else float('nan')

T = ['# STRATA, kernel do tipo D na velocidade do P (Llama-3.1-8B, llama.cpp CUDA)', '',
     f'GPU: {GPU}. Conferência do CUDA: {"tudo OK" if CUDA_OK else "DIVERGE"}. KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos do {TEXTO} de teste; '
     f'imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. Velocidade: tg128, mediana de {RODADAS} rodadas intercaladas de 5 repetições. '
     f'Antes: STRATA `{ANTES_SHA[:7]}`; depois: `{STRATA_SHA[:7]}`; llama.cpp `{LLAMA_SHA[:7]}`.', '',
     '## Velocidade de geração (tok/s)', '', '| modelo | depois | depois sem fusão | antes | rodadas (depois) |', '|---|---:|---:|---:|---|']
for m in ['Q5_K_M_imat', 'Q6_K_imat', 'P_N1', 'P_N2', 'D_N1', 'D_N2', 'DL_N1', 'DL_N2']:
    f = lambda k: f'{vel[k]:.1f}' if k in vel else '–'
    T.append(f'| {m} | {f(m)} | {f(m + "_sem_fusao")} | {f(m + "_antes")} | {", ".join(f"{x:.1f}" for x in rodadas[m])} |')
T += ['', '## Régua de velocidade', '', '| candidato | N1 / P | N1 / Q5_K_M | N2 / P | N2 / Q6_K | veredito |', '|---|---:|---:|---:|---:|---|']
for nome, pref, suf in [('D exato, depois', 'D', ''), ('D linha, depois', 'DL', ''), ('D exato, depois sem fusão', 'D', '_sem_fusao'),
                        ('D linha, depois sem fusão', 'DL', '_sem_fusao'), ('D exato, antes', 'D', '_antes')]:
    r = razoes(pref, suf)
    T.append(f'| {nome} | ' + ' | '.join(f'{x:.2f}' for x in r) + f' | **{veredito(r)}** |')
T += ['', '(Nas linhas "sem fusão" e "antes", o P é o do mesmo binário e modo; o k-quant é sempre o do "depois" com fusão.)']

T += ['', '## Qualidade', '', '| modelo | KL ×1000 (desquantização) | KL ×1000 (MMVQ, -ub 8) | mesmo 1º token (desq.) | GGUF (GiB) |', '|---|---:|---:|---:|---:|']
for m in ['Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'P_N1', 'D_N1', 'DL_N1', 'P_N2', 'D_N2', 'DL_N2']:
    a = f'{1000 * kld[m][0]:.3f} ± {1000 * kld[m][1]:.3f}' if m in kld else '–'
    b = f'{1000 * kld[m + "_mmvq"][0]:.3f} ± {1000 * kld[m + "_mmvq"][1]:.3f}' if m + '_mmvq' in kld else '–'
    T.append(f'| {m} | {a} | {b} | {kld[m][2] if m in kld else "–"}% | {tam.get(m, float("nan")):.3f} |')
T += ['', '| conferência | nível 1 | nível 2 | régua | resultado |', '|---|---:|---:|---|---|']
ok_mesma = []
cel = []
for k in (1, 2):
    a, b = kld.get(f'D_N{k}_mmvq_antes'), kld.get(f'D_N{k}_mmvq')
    if a and b:
        z = (b[0] - a[0]) / math.hypot(a[1], b[1])
        ok_mesma.append(abs(z) <= 1)
        cel.append(f'{1000 * a[0]:.3f} → {1000 * b[0]:.3f} ({z:+.2f} ±)')
    else:
        ok_mesma.append(False); cel.append('–')
T.append(f'| D antes → depois, MMVQ | {cel[0]} | {cel[1]} | dentro de 1 ± | **{"passa" if all(ok_mesma) else "reprova"}** |')
ok_l, cel, fr = [], [], []
for k in (1, 2):
    d, l, p = kl(f'D_N{k}'), kl(f'DL_N{k}'), kl(f'P_N{k}')
    ok_l.append(l <= 1.03 * d)
    cel.append(f'{l / d:.3f}')
    fr.append(f'{(p - l) / (p - d):.2f}' if p != d else '–')
T.append(f'| D linha / D, desquantização | {cel[0]} | {cel[1]} | ≤ 1,03 | **{"passa" if all(ok_l) else "reprova"}** |')
T.append(f'| fração do ganho do D sobre o P que o linha guarda | {fr[0]} | {fr[1]} | (1 = todo, 0 = nada) | |')

os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_kernel_d.md', 'w').write('\n'.join(T) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'cuda_ok': CUDA_OK, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS,
           'strata_sha': STRATA_SHA, 'antes_sha': ANTES_SHA, 'llama_sha': LLAMA_SHA,
           'kld': kld, 'gguf_gib': tam, 'vel_mediana': vel, 'vel_rodadas': rodadas},
          open('/content/res/strata_8b_kernel_d.json', 'w'), indent=1)
display(Markdown('\n'.join(T)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_kernel_d.md')
files.download('/content/res/strata_8b_kernel_d.json')